### Import

In [20]:
import re
import pickle 
import numpy as np
import pandas as pd
import datetime as dt

pd.set_option('display.max_columns', 500)

### General parameters

In [18]:
mimiciv = "PATH TO DATA/mimiciv(2.2)/"
output  = "../Data/csvExtract/"

### Read cohort hospital admission ids

In [3]:
cohort_subject_id_icu = []

with open("../Data/Cohort/cohort1_subject_id.txt", "r") as f:
    for subject_id in f:
        cohort_subject_id_icu.append(int(subject_id.strip()))
        
len(cohort_subject_id_icu)

16664

In [4]:
cohort_hadm_id_icu = []

with open("../Data/Cohort/cohort1_hadm_id.txt", "r") as f:
    for hadm_id in f:
        cohort_hadm_id_icu.append(int(hadm_id.strip()))
        
len(cohort_hadm_id_icu)

21454

In [5]:
cohort_stay_id_icu = []

with open("../Data/Cohort/cohort1_stay_id.txt", "r") as f:
    for stay_id in f:
        cohort_stay_id_icu.append(int(stay_id.strip()))
        
len(cohort_stay_id_icu)

23766

### hosp - patients

In [6]:
patients = pd.read_csv(mimiciv + "hosp/patients.csv")

In [7]:
patients.drop(columns=['anchor_year_group'], inplace=True)
patients['dod'] = pd.to_datetime(patients['dod'])
patients = patients[patients.subject_id.isin(cohort_subject_id_icu)]

In [8]:
gender_map = {'M': 1, 'F': 2}

def transform_gender(gender_series):
    return {'gender':gender_series.fillna('').apply(lambda s: gender_map[s] if s in gender_map else gender_map[''])}

patients.update(transform_gender(patients.gender))

In [9]:
patients.head(3)

In [10]:
print(patients.subject_id.nunique())
print(patients[patients.gender == 2].shape[0])
print(patients[patients.dod.notnull()].shape[0])

16664
7440
5367


### hosp - admission

In [11]:
admissions = pd.read_csv(mimiciv + "hosp/admissions.csv")

In [12]:
admissions.drop(columns=['admission_type', 'admit_provider_id', 'admission_location', 'discharge_location',
                         'insurance', 'language', 'marital_status', 'edregtime', 'edouttime'], inplace=True)

In [13]:
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
admissions['deathtime'] = pd.to_datetime(admissions['deathtime'])

In [14]:
def transform_race_into_id(df):
    
    df.race.fillna('nodx', inplace=True)
    dx_type = df.race.unique()
    dict_dx_key = pd.factorize(dx_type)[1]
    dict_dx_val = pd.factorize(dx_type)[0]
    dictionary  = dict(zip(dict_dx_key, dict_dx_val))
    df['race'] = df['race'].map(dictionary)
    
    return df, dictionary

In [15]:
admissions, race_dictionary = transform_race_into_id(admissions)

In [16]:
admissions = admissions[admissions.hadm_id.isin(cohort_hadm_id_icu)]

admissions = admissions[['subject_id', 'hadm_id', 'race', 'admittime', 'dischtime', 'deathtime',
                         'hospital_expire_flag']]

In [17]:
admissions.head(3)

In [18]:
print(admissions.hadm_id.nunique())
print(admissions.subject_id.nunique())
print(admissions[admissions.hospital_expire_flag == 1].shape[0])

21454
16664
2265


### hosp - merge admission and patients 

In [19]:
cohort_df = pd.merge(admissions, patients, on=['subject_id'], how='left')

In [20]:
cohort_df['death_time_disch'] = cohort_df['dod'] - cohort_df['dischtime']
cohort_df['death_time_disch'] = pd.to_timedelta(cohort_df['death_time_disch'])
cohort_df['death_time_disch'] = (cohort_df.death_time_disch / pd.Timedelta(days = 1)).round(0)
cohort_df['death_time_disch'] = cohort_df['death_time_disch'] + 1

In [21]:
cohort_df['age'] = (cohort_df['admittime'].dt.year - cohort_df['anchor_year']) + cohort_df['anchor_age']

cohort_df.drop(columns=['dod', 'anchor_year', 'anchor_age'], inplace=True)

cohort_df = cohort_df[['subject_id', 'hadm_id', 'gender', 'age', 'race', 'admittime', 'dischtime', 'deathtime', 
                       'hospital_expire_flag', 'death_time_disch']]

In [22]:
cohort_df.head(3)

In [23]:
print(cohort_df.hadm_id.nunique())
print(cohort_df.subject_id.nunique())
print(cohort_df[cohort_df.hospital_expire_flag == 1].shape[0])
print(cohort_df[cohort_df.death_time_disch.notnull()].shape[0])

21454
16664
2265
7855


### icu - icustays

In [24]:
icustays = pd.read_csv(mimiciv + "icu/icustays.csv")

In [25]:
icustays.drop(columns=['first_careunit', 'last_careunit'], inplace=True)

In [26]:
icustays['intime']  = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
icustays['los'] = icustays['los'].round(1)

In [27]:
icustays = icustays[icustays.stay_id.isin(cohort_stay_id_icu)]

In [28]:
icustays.head(3)

In [29]:
print(icustays.subject_id.nunique())
print(icustays.hadm_id.nunique())
print(icustays.stay_id.nunique())

16664
21454
23766


### merge cohort and icu

In [30]:
cohort_df = pd.merge(cohort_df, icustays, on=['subject_id', 'hadm_id'], how='left')

In [31]:
cohort_df['ADM_IN_DIFF'] = cohort_df['intime'] - cohort_df['admittime']
cohort_df['ADM_IN_DIFF'] = pd.to_timedelta(cohort_df['ADM_IN_DIFF'])
cohort_df['ADM_IN_DIFF'] = (cohort_df.ADM_IN_DIFF / pd.Timedelta(hours = 1)).round(1)

cohort_df['OUT_DIS_DIFF'] = cohort_df['dischtime'] - cohort_df['outtime']
cohort_df['OUT_DIS_DIFF'] = pd.to_timedelta(cohort_df['OUT_DIS_DIFF'])
cohort_df['OUT_DIS_DIFF'] = (cohort_df.OUT_DIS_DIFF / pd.Timedelta(hours = 1)).round(1)

In [32]:
cohort_df = cohort_df[~(((cohort_df.OUT_DIS_DIFF < -24) & (cohort_df.hospital_expire_flag == 0)))]

condition_1 = (cohort_df.ADM_IN_DIFF < 0)
cohort_df.loc[condition_1, 'admittime']  = cohort_df.loc[condition_1, 'intime']

condition_2 = ((cohort_df.OUT_DIS_DIFF < 0) & (cohort_df.hospital_expire_flag == 0))
cohort_df.loc[condition_2, 'dischtime']  = cohort_df.loc[condition_2, 'outtime']

cohort_df = cohort_df[cohort_df.OUT_DIS_DIFF > -48]

condition_3 = (cohort_df.OUT_DIS_DIFF < 0)
cohort_df.loc[condition_3, 'outtime']  = cohort_df.loc[condition_3, 'dischtime']

In [33]:
cohort_df.drop(columns=['OUT_DIS_DIFF', 'ADM_IN_DIFF'], inplace=True)

In [34]:
cohort_df = cohort_df[(cohort_df.stay_id.notnull()) & (cohort_df.intime >= cohort_df.admittime) & 
                      (cohort_df.outtime <= cohort_df.dischtime)]

In [35]:
cohort_df['icu_expire_flag'] = 0
cohort_df.loc[((cohort_df['deathtime'] > cohort_df['intime']) & (cohort_df['deathtime'] < cohort_df['outtime'] 
                                                              + pd.Timedelta(hours=6))), 'icu_expire_flag'] = 1

In [36]:
cohort_df['icuLos_h'] = cohort_df['outtime'] - cohort_df['intime']
cohort_df['icuLos_h'] = pd.to_timedelta(cohort_df['icuLos_h'])

cohort_df['icuLos_h'] = (cohort_df.icuLos_h / pd.Timedelta(hours = 1)).round(1)
cohort_df.rename(columns={"los": "icuLos_d"}, inplace=True)

In [37]:
cohort_df['hosp_Los'] = cohort_df['dischtime'] - cohort_df['admittime']
cohort_df['hosp_Los'] = pd.to_timedelta(cohort_df['hosp_Los'])

cohort_df['hospLos_d'] =  (cohort_df.hosp_Los / pd.Timedelta(days = 1)).round(1)
cohort_df['hospLos_h'] =  (cohort_df.hosp_Los / pd.Timedelta(hours= 1)).round(1)

cohort_df.drop(columns=['hosp_Los'], inplace=True)

In [38]:
cohort_df = cohort_df[['subject_id', 'hadm_id', 'stay_id', 'gender', 'age', 'race', 
                       'admittime', 'dischtime', 'intime', 'outtime', 'icuLos_h', 'icuLos_d',
                       'hospLos_h', 'hospLos_d', 'deathtime', 'icu_expire_flag', 'hospital_expire_flag',
                       'death_time_disch']]

In [39]:
cohort_df.head(3)

In [40]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())
print(cohort_df.stay_id.nunique())
print(cohort_df[cohort_df.icu_expire_flag == 1].shape[0])
print(cohort_df[cohort_df.hospital_expire_flag == 1].shape[0])
print(cohort_df[cohort_df.death_time_disch.notnull()].shape[0])

16664
21454
23766
1613
2683
8971


### Final cohort

In [41]:
cohort_icu_df = cohort_df[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']]

In [42]:
cohort_icu_df.head(3)

### Prescription

In [43]:
prescrip = []

for chunk in pd.read_csv(mimiciv + "hosp/prescriptions.csv", chunksize=10000):
    chunk = chunk[chunk.hadm_id.isin(cohort_hadm_id_icu)]
    chunk.drop(columns=['poe_id', 'poe_seq', 'order_provider_id', 'formulary_drug_cd',
                        'ndc', 'prod_strength', 'form_rx', 'form_val_disp', 'form_unit_disp',
                        'doses_per_24_hrs', 'pharmacy_id'], inplace=True)
    if chunk.shape[0]:
        prescrip.append(chunk)
        
prescriptions = pd.concat(prescrip)

In [44]:
antibiotic = prescriptions.copy()
antibiotic = antibiotic[antibiotic.drug_type != 'BASE']
antibiotic['drug'] = antibiotic['drug'].apply(str.lower)
antibiotic = antibiotic[~antibiotic.route.isin(['OU', 'OS', 'OD', 'AU', 'AS', 'AD', 'TP'])]
antibiotic = antibiotic[antibiotic.route.notnull()]
antibiotic['route'] = antibiotic['route'].apply(str.lower)
antibiotic = antibiotic[~antibiotic["route"].str.contains('eye|ear')]
antibiotic = antibiotic[~antibiotic["drug"].str.contains('cream|desensitization|ophth oint|gel')]

In [45]:
substring_antibiotics = ['adoxa', 'ala-tet', 'alodox', 'amikacin', 'amikin', 'amoxicill', 'amphotericin', 
                         'anidulafungin', 'ancef', 'clavulanate', 'ampicillin', 'augmentin', 'avelox', 'avidoxy', 
                         'azactam', 'azithromycin', 'aztreonam', 'axetil', 'bactocill', 'bactrim', 'bactroban', 
                         'bethkis', 'biaxin', 'bicillin l-a', 'cayston', 'cefazolin', 'cedax', 'cefoxitin', 
                         'ceftazidime', 'cefaclor', 'cefadroxil', 'cefdinir', 'cefditoren', 'cefepime', 'cefotan',
                         'cefotetan', 'cefotaxime', 'ceftaroline', 'cefpodoxime', 'cefpirome', 'cefprozil', 
                         'ceftibuten', 'ceftin', 'ceftriaxone', 'cefuroxime', 'cephalexin', 'cephalothin', 
                         'cephapririn', 'chloramphenicol', 'cipro', 'ciprofloxacin', 'claforan', 'clarithromycin',
                         'cleocin', 'clindamycin', 'cubicin', 'dicloxacillin', 'dirithromycin', 'doryx', 'doxycy',
                         'duricef', 'dynacin', 'ery-tab', 'eryped', 'eryc', 'erythrocin', 'erythromycin', 
                         'factive', 'flagyl', 'fortaz', 'furadantin', 'garamycin', 'gentamicin', 'kanamycin', 
                         'keflex', 'kefzol', 'ketek', 'levaquin', 'levofloxacin', 'lincocin', 'linezolid', 
                         'macrobid', 'macrodantin', 'maxipime', 'mefoxin', 'metronidazole', 'meropenem', 
                         'methicillin', 'minocin', 'minocycline', 'monodox', 'monurol', 'morgidox', 'moxatag', 
                         'moxifloxacin', 'mupirocin', 'myrac', 'nafcillin', 'neomycin', 'nicazel doxy 30',
                         'nitrofurantoin', 'norfloxacin', 'noroxin', 'ocudox', 'ofloxacin', 'omnicef', 'oracea', 
                         'oraxyl', 'oxacillin', 'pc pen vk', 'pce dispertab', 'panixine', 'pediazole', 'penicillin',
                         'periostat', 'pfizerpen', 'piperacillin', 'tazobactam', 'primsol', 'proquin', 'raniclor',
                         'rifadin', 'rifampin', 'rocephin', 'smz-tmp', 'septra', 'septra ds', 'septra', 'solodyn',
                         'spectracef', 'streptomycin', 'sulfadiazine', 'sulfamethoxazole', 'trimethoprim', 
                         'sulfatrim', 'sulfisoxazole', 'suprax', 'synercid', 'tazicef', 'tetracycline', 'timentin',
                         'tobramycin', 'trimethoprim', 'unasyn', 'vancocin', 'vancomycin', 'vantin', 'vibativ', 
                         'vibra-tabs', 'vibramycin', 'zinacef', 'zithromax', 'zosyn', 'zyvox']

In [46]:
gsn_antibiotics = [ '002542','002543','007371','008873','008877','008879','008880','008935',
                    '008941','008942','008943','008944','008983','008984','008990','008991',
                    '008992','008995','008996','008998','009043','009046','009065','009066',
                    '009136','009137','009162','009164','009165','009171','009182','009189',
                    '009213','009214','009218','009219','009221','009226','009227','009235',
                    '009242','009263','009273','009284','009298','009299','009310','009322',
                    '009323','009326','009327','009339','009346','009351','009354','009362',
                    '009394','009395','009396','009509','009510','009511','009544','009585',
                    '009591','009592','009630','013023','013645','013723','013724','013725',
                    '014182','014500','015979','016368','016373','016408','016931','016932',
                    '016949','018636','018637','018766','019283','021187','021205','021735',
                    '021871','023372','023989','024095','024194','024668','025080','026721',
                    '027252','027465','027470','029325','029927','029928','037042','039551',
                    '039806','040819','041798','043350','043879','044143','045131','045132',
                    '046771','047797','048077','048262','048266','048292','049835','050442',
                    '050443','051932','052050','060365','066295','067471']

In [47]:
antibiotic['value'] = 0

for string in substring_antibiotics:
    antibiotic.loc[antibiotic['drug'].str.contains(string),  'value'] = 1
    
antibiotic.loc[antibiotic['gsn'].isin(gsn_antibiotics),  'value'] = 1

antibiotic.loc[antibiotic.value == 1,  'drug'] = 'Antibiotic_PRC'
antibiotic = antibiotic[antibiotic.value == 1]

In [48]:
antibiotic.drop(columns=['gsn']   , inplace=True)
prescriptions.drop(columns=['gsn'], inplace=True)

In [49]:
antibiotic['starttime'] = pd.to_datetime(antibiotic['starttime'])
antibiotic['stoptime']  = pd.to_datetime(antibiotic['stoptime'])

In [50]:
antibiotic = pd.merge(antibiotic, cohort_icu_df, on=['subject_id', 'hadm_id'], how='left')
antibiotic = antibiotic[(antibiotic['starttime'] >= antibiotic['intime']) & (antibiotic['starttime'] <= antibiotic['outtime'])]
antibiotic.loc[(antibiotic['stoptime'] > antibiotic['outtime']), 'stoptime'] =  antibiotic['outtime']

In [51]:
Fentanyl = ['Fentanyl Citrate', 'Fentanyl Patch', 'Fentanyl PCA', 'Fentanyl', 'Fentanyl 100mcg/2mL 2mL AMP',
            'Fentanyl 2.5mg 50mL Bag', 'fentaNYL citrate (PF)', 'fentaNYL', 'fentaNYL Citrate', 
            'Fentanyl Intradermal', 'fentanyl', 'fentaNYL ', 'Actiq (Fentanyl Citrate)', 
            'fentaNYL IT 2000 mcg/ml pump', 'fentaNYL Citrate (PF)', 'fentaNYL citrate lollipops',
            'fentanyl 100 mcg/hr patch', 'fentaNYL Citrate (Actiq)', 'fentaNYL buccal', 'fentaNYL citrate',
            'fentaNYL citrate transmucosal lozenge', 'fentanyl citrate transmucosal lozenge', 'Fentanyl ']

Propofol = ['Propofol', 'Propofol 1000mg/100mL 100mL VIAL', 'Propofol 200mg/20mL 20mL VIAL', 
            'Propofol Intradermal ', 'propo', 'propofol (PF)', 'propox']

Norepinephrine = ['NORepinephrine', 'Norepinephrine', 'Norepinephrine Bitartrate', 'NORepinephrine (in NS)',
                  'NORepinephri 8mg/250mL 250mL BAG', 'norepinephrine bitartrate', 
                  'NORepinephrine ((for dil 8mg KIT', 'NORepinephrine ((for dilution)']


Insulin = ['Insulin', 'Insulin Human Regular', 'Insulin (Regular) for Hyperkalemia', 
           'Insulin Glargine  (CVICU Protocol)', 'Insulin Regular Human (U-500)', 
           'Insulin Pump (Self Administering Medication)', 'Insulin Syringe U-500', 'Insulin Pump',
           'Insulin Reg Human (NovoLIN R)', 'insulin detemir', 'Insulin Glargine', 'Insulin Lispro',
           'Insulin Aspart *NF*', 'NovoLOG U-100 Insulin aspart', 'Apidra SoloStar U-100 Insulin', 'insulin',
           'Toujeo SoloStar U-300 Insulin', 'insulin degludec', 'Insulin Detemir', 
           'Insulin Aspart *NF* (for Insulin PUMP)', 'Insulin (Regular) Intraperitoneal', 'Lantus U-100 Insulin',
           'insulin glargine', 'insulin lispro (for Insulin PUMP)', 'Insulin Human NPH', 'HumaLOG Insulin',
           'Insulin Levemir', 'Insulin (Regular)', 'Insulin (Regular) ', 'Levemir Insulin',
           'Novolog Flexpen U-100 Insulin', 'Humulin-R Insulin (U-500)', 'Fiasp FlexTouch U-100 Insulin',
           'HumuLIN N NPH U-100 Insulin', 'Insulin Regular Human (U-500) (for Insulin PUMP)', 'insulin glulisine',
           'Insulin Human 70/30', 'Insulin detemir', 'Insulin levimir', 'Insulin Regular Human', 'Insulin ',
           'Lantus Solostar U-100 Insulin', 'Insulin Novolog FlexPen', 'Insulin detemir (Levimir)', 
           'Apidra SoloStar U-100 Insulin    ** Non-Formulary **', 'insulin aspart', 'Humalog U-100 Insulin',
           'Insulin Lispro Desensitization Protocol', 'Insulin Detemir *NF*', 'Basaglar KwikPen U-100 Insulin',
           'Insulin NPH Human Recomb', 'Insulin levemir', 'Insulin degludec', 'NovoLIN N NPH U-100 Insulin', 
           'insulin aspart U-100', 'insulin ', ' Levemir Insulin', 'Levemir U-100 Insulin', 
           'NovoLOG Flexpen U-100 Insulin', 'Insulin Humalog 75/25 Pen', 'Insulin Asp Prt-Insulin Aspart',
           'Insulin-Symlin', 'Tresiba U-100 Insulin', 'HumaLOG KwikPen Insulin', 'Insulin lispro',
           'Humulin N NPH U-100 Insulin', 'insulin lispro protam-lispro', 'Insulin Degludec', 'insulin lispro',
           'Insulin (Regular', 'Levemir Insulin ', 'Insulin Levimir', 'Tresiba (Insulin degludec)',
           'Tresiba 100 units/ml (insulin deglutec)', 'Levemir insulin', 'sToujeo SoloStar U-300 Insulin', 
           'Insulin novolog', '*NF* Insulin Reg Human (NovoLIN R)', 'Insulin Detemir (Levemir)', 
           'Insulin Detemir (levemir)', 'Insulin ASPART', 'NPH insulin human recomb', 'Insulin detemir (levemir)',
           'NPH Insulin Human Recomb', 'HumuLIN N NPH Insulin KwikPen', 'insulin pump', 'Insulin Lispro 75/25',
           'Novolog U-100 Insulin aspart', 'Insulin TOUJEO', 'Insulin TUOJEO', 'Insulin Syringe U-100', 
           'Insulin Glargine ', 'Fiasp Penfill U-100 Insulin', 'Fiasp U-100 Insulin', 'Insulin degludec (Tresiba)',
           'Insulin 50/50', 'Tresiba FlexTouch U-100 *NF* (insulin degludec) ', 'HumaLOG U-100 Insulin',
           'Insulin-Levimir', 'Insulin-symlin', 'Insulin-levimir', 'Aspart Insulin Desensitization', 
           'Levemir (Insulin Detemir)', 'NovoLOG/Fiasp U-100 Insulin aspart', 'Insulin - Sliding Scale', 
           'insulin aspart ', 'insulin aspart (niacinamide)', 'Toujeo (glargine insulin) 300 units/ml',
           'Insulin Aspart', 'NovoLOG FlexPen U-100 Insulin', 'NovoLOG Mix 70-30 U-100 Insulin']


Midazolam = ['Midazolam', 'Midazolam 2mg/2mL 2mL Vial', 'Midazolam 100mg/100mL 100mL Bag', 'Midazolam Intradermal',
             'Midazolam ']

Heparin = ['Heparin', 'Heparin Sodium', 'Heparin Flush (10 units/ml)','Heparin Dwell (1000 Units/mL)',
           'Heparin Flush (100 units/ml)', 'Heparin (Hemodialysis)', 'Heparin Flush (1000 units/mL)',
           'Heparin (CRRT Machine Priming)', 'Heparin Flush PICC (100 units/ml)', 
           'Heparin Flush CVL  (100 units/ml)', 'Heparin Flush (5000 Units/mL)', 'Heparin (IABP)', 
           'Heparin (Impella)', 'Heparin Flush Port (10 units/mL)', 'Heparin Flush CRRT (5000 Units/mL)',
           'Heparin CRRT', 'Heparin Flush Midline (100 units/ml)', 'Heparin Flush Hickman (100 units/ml)',
           'Heparin INTRAPERITONEAL', 'Heparin ', 'heparin (porcine)', 'Heparin (For IP Use)', 
           'Heparin Flush (100 units/mL)', 'Heparin Flush (10 Units/mL)', 'Heparin Flush', 'Heparin Lock Flush',
           'Heparin Flush ', 'Heparin TOPICAL (100 Units/mL)', 'Heparin (Porcine)', 'heparin',
           'Hepari 25000UNIT/250mL 250mL BAG', 'Hepari', 'Heparin (Preservative Free)', 'Vancomycin-Heparin Lock',
           'HEParin', 'Heparin 1000UNIT/500mL 500mL BAG', 'Heparin 1000unit/1mL 10mL VIAL']

Dexmedetomidine = ['Dexmedetomidine', 'INV-Dexmedetomidine', 'Dexmedeto 0.4mg/100mL 100mL VIAL']

Amiodarone = ['Amiodarone', 'Amiodarone (f 150mg/3mL 3mL Vial', 'Amiodarone (for dilution)', 'Amiodarone ',
              'amiodarone', 'Amioda', 'Amiodarone Oral Suspension']

Vasopressin = ['Vasopressin', 'Vasopressin (for dilution)', 'Vasopressin (for 40units/2mL 2mL',]

Phenylephrine = ['PHENYLEPHrine', 'Phenylephrine', 'Phenylephrine 2.5 % Ophth Soln', 
                 'Phenylephrine  0.5% Nasal Spray', 'PHENYLEPHrine (for dilution)', 'PHENYLEPHrine (QuVa)', 
                 'PHENYLEPHrine (for 50mg/5mL VIAL', 'PHENYLEPHrin 50mg/250mL 250mL NS', 
                 'Phenyleprhine Ophth Soln 10%', 'PHENYLEPHrine 50mg/5mL 5mL VIAL', 
                 'PHENYLEPHrine (for dilu 60mg/6mL', 'phenylephrine HCl', 'Phenylephrine (For Priapism)',
                 'PHENYLEPHrin 60mg/250mL 250mL NS', 'phenylephrine', 'Cyclopentolate-Phenylephrine',
                 'Phenylephrine 10 % Ophth Soln', 'Phenylephrine 1% Nasal Spray', 'Neo-Synephrine (phenylephrine)',
                 'phenylephrin', 'phenyleph']

Epinephrine = ['EPINEPHrine', 'Epinephrine 1:1000', 'EPINEPHrine (EpiPEN)', 'Epinephrine',
               'Lidocaine 1%/Epinephrine 1:100000', 'Lidocaine 0.5%/Epinephrine',
               'Lidocaine 1%/Epinephrine 1:100,000', 'EPINEPHrine 1:1000', 'EPINEPHrine (for dilution)',
               'Lidocaine 2%/Epinephrine', 'EPINEPHrine (for dil 2mg/2mL 2mL', 'Epinephrine HCl', 'Epinephrine ',
               'Bupivacaine 0.50%-Epinephrine', 'Bupivacaine 0.25%-Epinephrine', 'Epinephrin',
               'Lidocaine 2%/Epinephrine P.F.', 'EPINEPHrine Auto Injector', 'Epinephri', 'Epinephr',
               'Epinephrine Auto Injector', 'EPINEPHrine 200mcg/25mL 25mL SYR', 'Lidocaine 1%/Epinephrine',
               'lidocaine-epinephrine (PF)', 'Epinephrine Topical Soln', 'Epineph', 
               'Lidocaine 1%/Epinephrine 1:200000', 'EPINEPHrine Kit', 'EPINEPHrine (B 1mg/10mL 10mL SYR',
               'EPINEPHrine ', 'Lidocaine 1.5%/Epinephrine P.F.', 'epinephrine', 'EPINEPHrine HCl (PF)',
               'Epinephrine Base', 'lidocaine-epinephrine', 'Epinephrine Kit', 'EPINEPHrine-Sodium Chloride',
               'Lidocaine 1%/Epinephrine ']

Dopamine = ['DOPamine', 'DOPamine 400mg BAG']

Nicardipine = ['NiCARdipine IV', 'NiCARdipine', 'Nicardipine', 'NiCARdipine 40mg/200mL 200mL BAG', 'niCARdipine']

Milrinone = ['Milrinone', 'Milrinone Lactate', 'milrinone']

Pantoprazole = ['Pantoprazole', 'Pantoprazole Sodium','Pantoprazole (Granules for DR Suspension)', 'pantoprazole',
                'Pantoprazole (Self Med)', 'Pantopr', 'Pantoprazol', 'pantoprazole oral solution', 
                'pantoprazole 2mg/ml', 'pantoprazole ', 'Pantoprazo', '*N F  Pantoprazole', 'Pantopra']

Diltiazem = ['Diltiazem', 'Diltiazem Extended-Release', 'Diltiaz', 'diltiaz', 'diltiazem', 'diltia', 'Diltia',
             'Dilti', 'Diltiazem ', 'dilti', 'Diltiaze', 'Diltiazem 25mg/5mL 5mL VIAL', 'diltiazem HCl',
             'Lidocaine 5% /diltiazem oint', 'diltiaze', 'DILTIAZEM', 'Diltiazem 50mg/10mL 10mL VIAL',
             '*NF* Diltiazem SR', 'Diltiazem HCl']

Dobutamine = ['DOBUTamine', 'DOBUTamine 250mg BAG']

Nitroglycerin = ['Nitroglycerin', 'Nitroglycerin SL', 'Nitroglycerin Ointment  2%', 'Nitroglycerin Patch',
                 'Nitroglycerin Ointment 0.2%', 'Nitroglycerin SR', 'nitroglycerin', 'Nitroglycerin Ointment ',
                 'Nitroglycerin SL 0.3mg BTL', 'Nitroglyceri', 'Nitroglyce 100mg/250mL 250mL BTL',
                 'nitroglycerin/lidocaine', 'Nitrolingual (Nitroglycerin)', 'Rectiv (nitroglycerin) 0.4%',
                 'Nitroglycerin 0.4% Ointment']

Warfarin    = ['Warfarin', 'Warf', 'Warfar', '*NF* Warfarin', 'warfarin', 'warf', 
               'warfarin (Coumadin) Brand Name', 'Coumadin']

Apixaban    = ['Apixaban', 'INV-Apixaban', 'apixaban', 'INV apixaban']

Dabigatran  = ['Dabigatran Etexilate', 'dabigatran etexilate', 'Dabigatran', 'dabigatran etexilate (Pradaxa)', 
               'Pradaxa']

Rivaroxaban = ['Rivaroxaban', 'rivaroxaban', 'INV-Rivaroxaban', 'Rivaro', 'Xarelto']

Edoxaban    = ['edoxaban', 'INV-Edoxaban', 'Edoxaban ', 'Edoxaban (Savaysa) 60mg tab', 'Edoxaban', 
               'INV-edoxaban', 'Edoxaban 60mg', 'Edoxiban']

In [52]:
prescriptions = prescriptions[prescriptions.drug.isin(Fentanyl)        | prescriptions.drug.isin(Propofol)     | 
                              prescriptions.drug.isin(Norepinephrine)  | prescriptions.drug.isin(Insulin)      | 
                              prescriptions.drug.isin(Midazolam)       | prescriptions.drug.isin(Heparin)      | 
                              prescriptions.drug.isin(Dexmedetomidine) | prescriptions.drug.isin(Amiodarone)   | 
                              prescriptions.drug.isin(Vasopressin)     | prescriptions.drug.isin(Phenylephrine)|
                              prescriptions.drug.isin(Dopamine)        | prescriptions.drug.isin(Nicardipine)  |
                              prescriptions.drug.isin(Milrinone)       | prescriptions.drug.isin(Pantoprazole) |
                              prescriptions.drug.isin(Diltiazem)       | prescriptions.drug.isin(Dobutamine)   |
                              prescriptions.drug.isin(Nitroglycerin)   | prescriptions.drug.isin(Epinephrine)  |
                              prescriptions.drug.isin(Warfarin)        | prescriptions.drug.isin(Apixaban)     | 
                              prescriptions.drug.isin(Dabigatran)      | prescriptions.drug.isin(Rivaroxaban)  |
                              prescriptions.drug.isin(Edoxaban)]

In [53]:
prescriptions.loc[prescriptions.drug.isin(Fentanyl)  ,       'drug'] = 'Fentanyl_PRC'
prescriptions.loc[prescriptions.drug.isin(Propofol)  ,       'drug'] = 'Propofol_PRC'
prescriptions.loc[prescriptions.drug.isin(Norepinephrine)  , 'drug'] = 'Norepinephrine_PRC'
prescriptions.loc[prescriptions.drug.isin(Insulin)  ,        'drug'] = 'Insulin_PRC'
prescriptions.loc[prescriptions.drug.isin(Midazolam)  ,      'drug'] = 'Midazolam_PRC'
prescriptions.loc[prescriptions.drug.isin(Heparin)  ,        'drug'] = 'Heparin_PRC'
prescriptions.loc[prescriptions.drug.isin(Dexmedetomidine),  'drug'] = 'Dexmedetomidine_PRC'
prescriptions.loc[prescriptions.drug.isin(Amiodarone)  ,     'drug'] = 'Amiodarone_PRC'
prescriptions.loc[prescriptions.drug.isin(Vasopressin)  ,    'drug'] = 'Vasopressin_PRC'
prescriptions.loc[prescriptions.drug.isin(Phenylephrine)  ,  'drug'] = 'Phenylephrine_PRC'
prescriptions.loc[prescriptions.drug.isin(Dopamine)  ,       'drug'] = 'Dopamine_PRC'
prescriptions.loc[prescriptions.drug.isin(Nicardipine)  ,    'drug'] = 'Nicardipine_PRC'
prescriptions.loc[prescriptions.drug.isin(Milrinone)  ,      'drug'] = 'Milrinone_PRC'
prescriptions.loc[prescriptions.drug.isin(Pantoprazole)  ,   'drug'] = 'Pantoprazole_PRC'
prescriptions.loc[prescriptions.drug.isin(Diltiazem)  ,      'drug'] = 'Diltiazem_PRC'
prescriptions.loc[prescriptions.drug.isin(Dobutamine)  ,     'drug'] = 'Dobutamine_PRC'
prescriptions.loc[prescriptions.drug.isin(Nitroglycerin)  ,  'drug'] = 'Nitroglycerin_PRC'
prescriptions.loc[prescriptions.drug.isin(Epinephrine)  ,    'drug'] = 'Epinephrine_PRC'
prescriptions.loc[prescriptions.drug.isin(Warfarin),         'drug'] = 'Warfarin_PRC'
prescriptions.loc[prescriptions.drug.isin(Apixaban),         'drug'] = 'Apixaban_PRC'
prescriptions.loc[prescriptions.drug.isin(Dabigatran),       'drug'] = 'Dabigatran_PRC'
prescriptions.loc[prescriptions.drug.isin(Rivaroxaban),      'drug'] = 'Rivaroxaban_PRC'
prescriptions.loc[prescriptions.drug.isin(Edoxaban),         'drug'] = 'Edoxaban_PRC'

In [54]:
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])
prescriptions['stoptime']  = pd.to_datetime(prescriptions['stoptime'])

In [55]:
prescriptions = pd.merge(prescriptions, cohort_icu_df, on=['subject_id', 'hadm_id'], how='left')
prescriptions = prescriptions[(prescriptions['starttime'] >= prescriptions['intime']) & (prescriptions['starttime'] <= prescriptions['outtime'])]
prescriptions.loc[(prescriptions['stoptime'] > prescriptions['outtime']), 'stoptime'] =  prescriptions['outtime']

In [56]:
antibiotic = antibiotic[['subject_id', 'hadm_id', 'stay_id', 'starttime', 'stoptime', 'drug']]
antibiotic.rename(columns={"drug": "label"}, inplace=True)

prescriptions = prescriptions[['subject_id', 'hadm_id', 'stay_id', 'starttime', 'stoptime', 'drug']]
prescriptions.rename(columns={"drug": "label"}, inplace=True)

In [57]:
antibiotic['Hour_diff'] = (antibiotic.stoptime - antibiotic.starttime).apply(lambda s: s / np.timedelta64(1, 's')) / 60./60
prescriptions['Hour_diff'] = (prescriptions.stoptime - prescriptions.starttime).apply(lambda s: s / np.timedelta64(1, 's')) / 60./60

antibiotic.loc[antibiotic.Hour_diff < 0,    'stoptime']  = antibiotic['starttime']
antibiotic.loc[antibiotic.Hour_diff < 0,    'Hour_diff'] = 0

prescriptions.loc[prescriptions.Hour_diff < 0,    'stoptime']  = antibiotic['starttime']
prescriptions.loc[prescriptions.Hour_diff < 0,    'Hour_diff'] = 0

antibiotic.drop(columns=['Hour_diff', 'stoptime']   , inplace=True)
prescriptions.drop(columns=['Hour_diff', 'stoptime'], inplace=True)

antibiotic['value'] = 1
prescriptions['value'] = 1

In [58]:
prescriptions_list = [antibiotic, prescriptions]
prescriptions = pd.concat(prescriptions_list)
prescriptions.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']

In [59]:
prescriptions.head(2)

In [60]:
print(prescriptions.subject_id.nunique())
print(prescriptions.hadm_id.nunique())
print(prescriptions.stay_id.nunique())

15949
20438
22348


### microbiology events

In [61]:
microbioevents = []

for chunk in pd.read_csv(mimiciv + "hosp/microbiologyevents.csv", chunksize=10000):
    chunk = chunk[chunk.subject_id.isin(cohort_subject_id_icu)]
    chunk = chunk[['subject_id', 'hadm_id', 'chartdate', 'charttime', 'ab_name','interpretation']]
    if chunk.shape[0] > 0:
        microbioevents.append(chunk)
        
microbio_df = pd.concat(microbioevents)

In [62]:
microbio_df['chartdate'] = pd.to_datetime(microbio_df['chartdate'])
microbio_df['charttime'] = pd.to_datetime(microbio_df['charttime'])

In [63]:
microbio_df.loc[microbio_df['charttime'].isnull(),  'charttime'] = microbio_df['chartdate']
microbio_df.loc[microbio_df['ab_name'].isnull(),  'ab_name'] = 'NO GROWTH'
microbio_df.loc[microbio_df['interpretation'].isnull(),  'interpretation'] = 'N'
microbio_df.drop(columns=['chartdate'], inplace=True)

In [64]:
microbio_df = pd.merge(microbio_df, cohort_icu_df, on=['subject_id'], how='left')

microbio_df = microbio_df[(microbio_df['charttime'] >= microbio_df['intime']) & 
                          (microbio_df['charttime'] <= microbio_df['outtime'])]

In [65]:
microbio_df = microbio_df[['subject_id', 'hadm_id_y', 'stay_id', 'charttime', 'ab_name','interpretation']]
microbio_df.rename(columns={"hadm_id_y":"hadm_id", "ab_name": "label", "interpretation": "value"}, inplace=True)

In [66]:
microbio_df['label'] = 'Microbio Test'
microbio_df.loc[microbio_df['value'] == 'N'  , 'value'] = -1
microbio_df.loc[microbio_df['value'] == 'S'  , 'value'] = 4
microbio_df.loc[microbio_df['value'] == 'R'  , 'value'] = 3
microbio_df.loc[microbio_df['value'] == 'I'  , 'value'] = 2
microbio_df.loc[microbio_df['value'] == 'P'  , 'value'] = 1

In [67]:
microbio_df.head(3)

In [68]:
print(microbio_df.subject_id.nunique())
print(microbio_df.hadm_id.nunique())
print(microbio_df.stay_id.nunique())

12946
16280
17561


### lab item ids 

In [69]:
d_labitems = pd.read_csv(mimiciv + "hosp/d_labitems.csv")
d_labitems.drop(columns=['fluid', 'category'], inplace=True)

### lab event

In [70]:
def check(x):
    try:
        x = float(str(x).strip())
    except ValueError:
        try:
            x = float(re.findall(r'\d+', x)[0])
        except IndexError:
            x = np.nan
    return x

def check_itemvalue(df):
    df['value'] = df['value'].apply(lambda x: check(x))
    return df

In [71]:
labevents = []

for chunk in pd.read_csv(mimiciv + "hosp/labevents.csv", chunksize=10000):
    chunk = chunk[chunk.hadm_id.isin(cohort_hadm_id_icu)]
    chunk = chunk[['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum', 'valueuom']]
    chunk = chunk[chunk.itemid.notnull()]
    chunk = chunk[(chunk.value.notnull()) | (chunk.valuenum.notnull())]
    if chunk.shape[0] > 0:
        labevents.append(chunk)
        
lab_df = pd.concat(labevents)

In [72]:
lab_df = pd.merge(lab_df, d_labitems, on=['itemid'], how='left')
lab_df = pd.merge(lab_df, cohort_icu_df, on=['subject_id', 'hadm_id'], how='left')

In [73]:
lab_df['charttime'] = pd.to_datetime(lab_df['charttime'])
lab_df = lab_df[(lab_df['charttime'] >= lab_df['intime']) & (lab_df['charttime'] <= lab_df['outtime'])]

In [74]:
lab_df = lab_df[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value', 'valuenum', 'valueuom']]

In [75]:
lab_df['string'] = lab_df['value'].str.extract('([A-Za-z]+)')
lab_df.loc[(lab_df.string.notnull()) & (lab_df.valuenum.notnull()), 'value'] = lab_df['valuenum']
lab_df['string'] = lab_df['value'].str.extract('([A-Za-z]+)')

In [76]:
lab_df = check_itemvalue(lab_df)

In [77]:
lab_df.loc[(lab_df.value != lab_df.valuenum) & (lab_df.value.isnull()) & (lab_df.valuenum.notnull()), 'value'] = lab_df['valuenum']
lab_df.loc[((lab_df.string.notnull()) & (lab_df.value.isnull())), 'value'] = lab_df['string']
lab_df.loc[(lab_df['label'] == 'Absolute Lymphocyte Count') & (lab_df['valueuom'] == '#/uL'), 'value'] =  np.nan

In [78]:
Intubated_0 = ['NOT']
Intubated_1 = ['INTUBATED']

lab_df.loc[(lab_df['label'] == 'Intubated') & (lab_df['value'] != 'NOT') & (lab_df['value'] != 'INTUBATED'),  'value'] = np.nan
lab_df.loc[(lab_df['label'] == 'Intubated') & (lab_df['value'].isin(Intubated_0)),  'value'] = 0
lab_df.loc[(lab_df['label'] == 'Intubated') & (lab_df['value'].isin(Intubated_1)),  'value'] = 1

In [79]:
lab_df = lab_df[lab_df.label.notnull()]
lab_df = lab_df[lab_df.value.notnull()]
lab_df.drop(columns=['valuenum', 'string', 'valueuom'], inplace=True)

In [80]:
lab_df.head(3)

In [81]:
print(lab_df.subject_id.nunique())
print(lab_df.hadm_id.nunique())
print(lab_df.stay_id.nunique())

16381
21059
23184


### icu - item ids

In [82]:
d_icuitems = pd.read_csv(mimiciv + "icu/d_items.csv")
d_icuitems.drop(columns=['abbreviation', 'category', 'unitname', 'param_type',
                         'lownormalvalue', 'highnormalvalue'], inplace=True)

### icu - chartevent

In [83]:
chartevent = []

for chunk in pd.read_csv(mimiciv + "icu/chartevents.csv", chunksize=10000):
    chunk = chunk[chunk.stay_id.isin(cohort_stay_id_icu)]
    chunk = chunk[['subject_id', 'hadm_id', 'stay_id', 'itemid', 'charttime', 'value', 'valuenum', 'valueuom']]
    chunk = chunk[chunk.itemid.notnull()]
    chunk = chunk[(chunk.value.notnull()) | (chunk.valuenum.notnull())]
    if chunk.shape[0] > 0:
        chartevent.append(chunk)
        
chart_df = pd.concat(chartevent)

In [84]:
chart_df = pd.merge(chart_df, d_icuitems, on=['itemid'], how='left')
chart_df['charttime'] = pd.to_datetime(chart_df['charttime'])
chart_df = chart_df[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value', 'valuenum', 'valueuom']]

In [85]:
chart_df['string'] = chart_df['value'].str.extract('([A-Za-z]+)')
chart_df.loc[(chart_df.string.notnull()) & (chart_df.valuenum.notnull()), 'value'] = chart_df['valuenum']
chart_df['string'] = chart_df['value'].str.extract('([A-Za-z]+)')
chart_df.loc[(chart_df.string.notnull()), 'string'] = chart_df['value']

In [86]:
chart_df = check_itemvalue(chart_df)
chart_df.loc[(chart_df.value != chart_df.valuenum) & (chart_df.value.isnull()) & (chart_df.valuenum.notnull()), 'value'] = chart_df['valuenum']
chart_df.loc[((chart_df.string.notnull()) & (chart_df.value.isnull())), 'value'] = chart_df['string']

In [87]:
MentalStatus_1 = [15.0]

chart_df.loc[(chart_df['label'] == 'Mental status') & (chart_df['value'] != 0.0) & (chart_df['value'] != 15.0),  'value'] = np.nan
chart_df.loc[(chart_df['label'] == 'Mental status') & (chart_df['value'].isin(MentalStatus_1)),    'value'] = 1.0

In [88]:
PainPresent_0 = ['No']
PainPresent_1 = ['Yes']

chart_df.loc[(chart_df['label'] == 'Pain Present') & (chart_df['value'] != 'No') & (chart_df['value'] != 'Yes'),  'value'] = np.nan
chart_df.loc[(chart_df['label'] == 'Pain Present') & (chart_df['value'].isin(PainPresent_0)),  'value'] = 0
chart_df.loc[(chart_df['label'] == 'Pain Present') & (chart_df['value'].isin(PainPresent_1)),  'value'] = 1

In [89]:
DeliriumAssess_0 = ['Negative']
DeliriumAssess_1 = ['Positive']
DeliriumAssess_nan = ['UTA']

chart_df.loc[(chart_df['label'] == 'Delirium assessment') & (chart_df['value'] != 'Negative') & (chart_df['value'] != 'Positive'),  'value'] = np.nan
chart_df.loc[(chart_df['label'] == 'Delirium assessment') & (chart_df['value'].isin(DeliriumAssess_0)),    'value'] = 0
chart_df.loc[(chart_df['label'] == 'Delirium assessment') & (chart_df['value'].isin(DeliriumAssess_1)),    'value'] = 1
chart_df.loc[(chart_df['label'] == 'Delirium assessment') & (chart_df['value'].isin(DeliriumAssess_nan)),  'value'] = np.nan

In [90]:
CAMICU_RASS_LOC_0 = ['No']
CAMICU_RASS_LOC_1 = ['Yes']

chart_df.loc[(chart_df['label'] == 'CAM-ICU RASS LOC') & (chart_df['value'] != 'No') & (chart_df['value'] != 'Yes'),  'value'] = np.nan
chart_df.loc[(chart_df['label'] == 'CAM-ICU RASS LOC') & (chart_df['value'].isin(CAMICU_RASS_LOC_0)),    'value'] = 0
chart_df.loc[(chart_df['label'] == 'CAM-ICU RASS LOC') & (chart_df['value'].isin(CAMICU_RASS_LOC_1)),    'value'] = 1

In [91]:
SkinTemperature_1 = ['Cold']
SkinTemperature_2 = ['Cool']
SkinTemperature_3 = ['Warm']
SkinTemperature_4 = ['Hot']

chart_df.loc[(chart_df['label'] == 'Skin Temperature') & (chart_df['value'] != 'Cold') & 
             (chart_df['value'] != 'Cool') & (chart_df['value'] != 'Warm') & (chart_df['value'] != 'Hot'),  'value'] = np.nan

chart_df.loc[(chart_df['label'] == 'Skin Temperature') & (chart_df['value'].isin(SkinTemperature_1)), 'value'] = 1
chart_df.loc[(chart_df['label'] == 'Skin Temperature') & (chart_df['value'].isin(SkinTemperature_2)), 'value'] = 2
chart_df.loc[(chart_df['label'] == 'Skin Temperature') & (chart_df['value'].isin(SkinTemperature_3)), 'value'] = 3
chart_df.loc[(chart_df['label'] == 'Skin Temperature') & (chart_df['value'].isin(SkinTemperature_4)), 'value'] = 4

In [92]:
RiskFalls_0 = ['No']
RiskFalls_1 = ['Yes']

chart_df.loc[(chart_df['label'] == 'Risk for Falls') & (chart_df['value'] != 'No') & (chart_df['value'] != 'Yes'),  'value'] = np.nan
chart_df.loc[(chart_df['label'] == 'Risk for Falls') & (chart_df['value'].isin(RiskFalls_0)),    'value'] = 0
chart_df.loc[(chart_df['label'] == 'Risk for Falls') & (chart_df['value'].isin(RiskFalls_1)),    'value'] = 1

In [93]:
CAMICU_MS_change_0 = ['No (Stop - Not delirious)']
CAMICU_MS_change_1 = ['Yes (Continue)']
CAMICU_MS_change_nan = ['Unable to Assess (Stop)']

chart_df.loc[(chart_df['label'] == 'CAM-ICU MS change') & (chart_df['value'] != 'No (Stop - Not delirious)') & 
             (chart_df['value'] != 'Yes (Continue)'),  'value'] = np.nan
chart_df.loc[(chart_df['label'] == 'CAM-ICU MS change') & (chart_df['value'].isin(CAMICU_MS_change_0)),   'value'] = 0
chart_df.loc[(chart_df['label'] == 'CAM-ICU MS change') & (chart_df['value'].isin(CAMICU_MS_change_1)),   'value'] = 1
chart_df.loc[(chart_df['label'] == 'CAM-ICU MS change') & (chart_df['value'].isin(CAMICU_MS_change_nan)), 'value'] = np.nan

In [94]:
CAMICU_Disorganized_thinking_0 = ['No']
CAMICU_Disorganized_thinking_1 = ['Yes']

chart_df.loc[(chart_df['label'] == 'CAM-ICU Disorganized thinking') & (chart_df['value'].isin(CAMICU_Disorganized_thinking_0)),    'value'] = 0
chart_df.loc[(chart_df['label'] == 'CAM-ICU Disorganized thinking') & (chart_df['value'].isin(CAMICU_Disorganized_thinking_1)),    'value'] = 1

In [95]:
chart_df = chart_df[chart_df.label.notnull()]
chart_df = chart_df[chart_df.value.notnull()]
chart_df.drop(columns=['valuenum', 'string', 'valueuom'], inplace=True)

### culture <== chartevent

In [96]:
culture_list = ['Arterial Line Tip Cultured', 'CCO PAC Line Tip Cultured', 'Cordis/Introducer Line Tip Cultured',
                'Dialysis Catheter Tip Cultured', 'Tunneled (Hickman) Line Tip Cultured', 'IABP Line Tip Cultured',
                'Midline Tip Cultured', 'Multi Lumen Line Tip Cultured', 'PA Catheter Line Tip Cultured',
                'Pheresis Catheter Line Tip Cultured', 'PICC Line Tip Cultured', 'Trauma Line Tip Cultured',
                'Indwelling Port (PortaCath) Line Tip Cultured', 'Presep Catheter Line Tip Cultured', 
                'AVA Line Tip Cultured', 'Triple Introducer Line Tip Cultured', 'Sheath Line Tip Cultured',
                'ICP Line Tip Cultured']

In [97]:
culture_df = chart_df[ chart_df.label.isin(culture_list)]
chart_df   = chart_df[~chart_df.label.isin(culture_list)]

culture_df['label'] = 'Blood Culture'

In [98]:
chart_df.head(3)

In [99]:
print(chart_df.subject_id.nunique())
print(chart_df.hadm_id.nunique())
print(chart_df.stay_id.nunique())

16664
21454
23765


In [100]:
culture_df.head(3)

In [101]:
print(culture_df.subject_id.nunique())
print(culture_df.hadm_id.nunique())
print(culture_df.stay_id.nunique())

2175
2319
2394


### icu - datetiemevent

In [102]:
dateevents = []

for chunk in pd.read_csv(mimiciv + "icu/datetimeevents.csv", chunksize=10000):
    chunk = chunk[chunk.stay_id.isin(cohort_stay_id_icu)]
    chunk.drop(columns=['caregiver_id', 'storetime', 'valueuom', 'warning'], inplace=True)

    if chunk.shape[0] > 0:
        dateevents.append(chunk)
        
datetimeevents = pd.concat(dateevents)

In [103]:
datetimeevents = pd.merge(datetimeevents, d_icuitems, on=['itemid'], how='left')
datetimeevents = datetimeevents[['subject_id', 'hadm_id', 'stay_id', 'value', 'label']]
datetimeevents['value'] = pd.to_datetime(datetimeevents['value'], errors='coerce')
datetimeevents = datetimeevents[datetimeevents.value.notnull()]
datetimeevents.rename(columns={"value": "charttime"}, inplace=True)
datetimeevents['value'] = 1

In [104]:
date_var = ['22 Gauge Insertion Date', '20 Gauge Insertion Date', '18 Gauge Insertion Date',
            'Arterial line Insertion Date', 'Arterial line Tubing Change', 'Arterial Line Dressing Change',
            'Multi Lumen Insertion Date', 'Multi Lumen Cap Change', 'Multi Lumen Dressing Change',
            'Multi Lumen Tubing Change']

datetimeevents = datetimeevents[datetimeevents.label.isin(date_var)]

In [105]:
datetimeevents.head(3)

In [106]:
print(datetimeevents.subject_id.nunique())
print(datetimeevents.hadm_id.nunique())
print(datetimeevents.stay_id.nunique())
print(datetimeevents.label.nunique())

16380
20940
22789
10


### icu - outputevent

In [107]:
outevents = []

for chunk in pd.read_csv(mimiciv + "icu/outputevents.csv", chunksize=10000):
    chunk = chunk[chunk.stay_id.isin(cohort_stay_id_icu)]

    if chunk.shape[0] > 0:
        outevents.append(chunk)
        
outputevents = pd.concat(outevents)

In [108]:
outputevents = pd.merge(outputevents, d_icuitems, on=['itemid'], how='left')
outputevents['charttime'] = pd.to_datetime(outputevents['charttime'])
outputevents = outputevents[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']]

In [109]:
Urine_output = ['R Ureteral Stent', 'L Ureteral Stent', 'Foley', 'Void', 'Condom Cath', 'Suprapubic', 
                'R Nephrostomy', 'L Nephrostomy', 'Ileoconduit', 'GU Irrigant/Urine Volume Out', 'OR Urine']

Stool = ['Stool']

In [110]:
outputevents = outputevents[outputevents.label.isin(Urine_output) | outputevents.label.isin(Stool)]

In [111]:
outputevents.loc[outputevents.label.isin(Urine_output), 'label'] = 'UrineOutput_IO'
outputevents.loc[outputevents.label.isin(Stool),        'label'] = 'Stool_IO'

In [112]:
outputevents.head(3)

In [113]:
print(outputevents.subject_id.nunique())
print(outputevents.hadm_id.nunique())
print(outputevents.stay_id.nunique())
print(outputevents.label.nunique())

16340
20778
22871
2


### icu - inputevent

In [114]:
inputevent = []

for chunk in pd.read_csv(mimiciv + "icu/inputevents.csv", chunksize=10000):
    chunk = chunk[chunk.stay_id.isin(cohort_stay_id_icu)]

    if chunk.shape[0] > 0:
        inputevent.append(chunk)
        
inputevents = pd.concat(inputevent)

In [115]:
inputevents = pd.merge(inputevents, d_icuitems, on=['itemid'], how='left')
inputevents['starttime'] = pd.to_datetime(inputevents['starttime'])
inputevents['endtime']   = pd.to_datetime(inputevents['endtime'])

In [116]:
inputevents['duration'] = inputevents['endtime'] - inputevents['starttime']
inputevents['duration'] = pd.to_timedelta(inputevents['duration'])
inputevents['duration'] = (inputevents.duration / pd.Timedelta(hours = 1)).round(0)
inputevents = inputevents[['subject_id', 'hadm_id', 'stay_id', 'starttime', 'endtime', 
                           'itemid', 'label', 'amount', 'amountuom']]

In [117]:
inputevents.loc[(inputevents.label == 'Ceftriaxone')   & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Cefazolin')     & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Cefepime')      & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Ceftazidime')   & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Vancomycin')    & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Clindamycin')   & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Metronidazole') & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Meropenem')     & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Tobramycin')    & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Acyclovir')     & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Linezolid')     & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Azithromycin')  & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Voriconazole')  & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Levofloxacin')  & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Micafungin')    & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Fluconazole')   & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Ranitidine (Prophylaxis)') & (inputevents.amount.notnull()), 'amount'] = 1
inputevents.loc[(inputevents.label == 'Famotidine (Pepcid)')      & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[(inputevents.label == 'Pantoprazole (Protonix)')  & (inputevents.amount.notnull()), 'amount'] = 1 
inputevents.loc[((inputevents.label == 'Propofol')          & (inputevents.amountuom == 'mcg')) , 'amount'] = np.nan 
inputevents.loc[((inputevents.label == 'Ciprofloxacin')     & (inputevents.amountuom == 'mg'))  , 'amount'] = np.nan 
inputevents.loc[((inputevents.label == 'Fentanyl')          & (inputevents.amountuom == 'mcg')) , 'amount'] = inputevents['amount']/1000
inputevents.loc[((inputevents.label == 'Digoxin (Lanoxin)') & (inputevents.amountuom == 'mcg')) , 'amount'] = inputevents['amount']/1000
inputevents.loc[((inputevents.label == 'Thiamine')          & (inputevents.amountuom == 'dose')), 'amount'] = inputevents['amount']*1000 
inputevents.loc[((inputevents.label == 'Beneprotein')       & (inputevents.amountuom == 'dose')), 'amount'] = inputevents['amount']*100 
inputevents.loc[((inputevents.label == 'Digoxin (Lanoxin)') & (inputevents.amountuom == 'mg'))  , 'amount'] = inputevents['amount']*1000 
inputevents.loc[((inputevents.label == 'Fentanyl (Concentrate)')       & (inputevents.amountuom == 'mcg')), 'amount'] = inputevents['amount']/1000
inputevents.loc[((inputevents.label == 'Pre-Admission/Non-ICU Intake') & (inputevents.amountuom == 'L'))  , 'amount'] = inputevents['amount']*1000 

In [118]:
Dobutamine_ID = [221653]
Milrinone_ID  = [221986]
NeuroblockAgent_ID = [222062, 221555]
KIV_ID = [225166, 225834, 222139, 225925]
CaIV_ID = [228317, 229640]
CanonIV_ID = [221456, 227525, 229618]
MgIV_ID = [227524]
MgnonIV_ID = [227523, 222011]
PIV_ID = [225925, 225834]
PnonIV_ID = [225890]
Fluids_ID = [221212, 221213, 225161, 226401]
BetaBlockers_ID = [225974, 221429]
CaBlockers_ID = [221347, 229654, 228339, 221468]
LoopDiuretics_ID = [221794, 228340]
TPNutrition_ID = [225916, 225917]
Fentanyl_ID = [225972, 225942, 221744]
Albumin_ID = [220861, 220862, 220863, 220864]
Insulin_ID = [223257, 223258, 223259, 223260, 223261, 223262, 229299, 229619]
Dextrose_ID = [220949, 220950, 220951, 220952, 220963, 220964, 220965, 220966, 220967, 220968,
               221000, 221002, 221014, 221017, 228140, 228141, 228142] 
PNutrition_ID = [227090, 222190, 225801, 225916, 225917, 225920, 225947, 225948, 225969]
POnutrition_ID = [221036, 225931, 226880, 227518, 226017, 226016, 226018, 226019, 226028, 226029, 226030, 226881]
Vasopressors_ID = [222315, 221289, 221906, 229617, 221749, 229630, 221662, 229632, 229631]    

In [119]:
Propofol = ['Propofol']

Fentanyl = ['Fentanyl', 'Fentanyl (Concentrate)']

Insulin = ['Insulin - Regular', 'Insulin - Humalog', 'Insulin - Glargine', 'Insulin - NPH', 'Insulin - Novolog',
           'Insulin - 70/30', 'Insulin - Humalog 75/25', 'Insulin - U500']

Heparin = ['Heparin Sodium (Prophylaxis)', 'Heparin Sodium', 'Heparin Sodium (Impella)']

Midazolam = ['Midazolam (Versed)']

Dexmedetomidine = ['Dexmedetomidine (Precedex)']

Vassopressin = ['Vasopressin']

Albumin = ['Albumin 5%', 'Albumin 25%']

Ceftriaxone = ['Ceftriaxone']

Cefazolin = ['Cefazolin']

Cefepime = ['Cefepime']

Ceftazidime = ['Ceftazidime']

Vancomycin = ['Vancomycin']

Clindamycin = ['Clindamycin']

Metronidazole = ['Metronidazole']

Meropenem = ['Meropenem']

Acyclovir = ['Acyclovir']

Azithromycin = ['Azithromycin']

Levofloxacin = ['Levofloxacin']

Micafungin = ['Micafungin']

Fluconazole = ['Fluconazole']

Thiamine = ['Thiamine']

Dobutamine = ['Dobutamine']

Milrinone = ['Milrinone']

PO = ['PO Intake']

Crystalloids = ['OR Crystalloid Intake']

Norepinephrine = ['Norepinephrine']

Amiodarone = ['Amiodarone', 'Amiodarone 600/500', 'Amiodarone 450/250']

Phenylephrine = ['Phenylephrine', 'Phenylephrine (50/250)', 'Phenylephrine (200/250)']

Epinephrine = ['Epinephrine', 'Epinephrine.']

Nicardipine = ['Nicardipine', 'Nicardipine 40mg/200']

Pantoprazole = ['Pantoprazole (Protonix)']

Diltiazem = ['Diltiazem']

Nitroglycerin = ['Nitroglycerin']

SodiumChloride = []

IVPB = []

Fluids = []

OralIntake = []

NSIVF = []

In [120]:
inputevents = inputevents[inputevents.itemid.isin(Dobutamine_ID)       | inputevents.itemid.isin(Milrinone_ID)    |
                          inputevents.itemid.isin(NeuroblockAgent_ID)  | inputevents.itemid.isin(KIV_ID)          |
                          inputevents.itemid.isin(CaIV_ID)             | inputevents.itemid.isin(CanonIV_ID)      |
                          inputevents.itemid.isin(MgIV_ID)             | inputevents.itemid.isin(MgnonIV_ID)      |
                          inputevents.itemid.isin(PIV_ID)              | inputevents.itemid.isin(PnonIV_ID)       |
                          inputevents.itemid.isin(Fluids_ID)           | inputevents.itemid.isin(BetaBlockers_ID) |
                          inputevents.itemid.isin(CaBlockers_ID)       | inputevents.itemid.isin(LoopDiuretics_ID)|
                          inputevents.itemid.isin(TPNutrition_ID)      | inputevents.itemid.isin(Fentanyl_ID)     |
                          inputevents.itemid.isin(Albumin_ID)          | inputevents.itemid.isin(Insulin_ID)      |
                          inputevents.itemid.isin(Dextrose_ID)         | inputevents.itemid.isin(PNutrition_ID)   |
                          inputevents.itemid.isin(POnutrition_ID)      | inputevents.itemid.isin(Vasopressors_ID) |

                          inputevents.label.isin(Propofol)             | inputevents.label.isin(Fentanyl)         |
                          inputevents.label.isin(Insulin)              | inputevents.label.isin(Heparin)          |
                          inputevents.label.isin(Midazolam)            | inputevents.label.isin(Dexmedetomidine)  |
                          inputevents.label.isin(Vassopressin)         | inputevents.label.isin(Albumin)          |
                          inputevents.label.isin(Ceftriaxone)          | inputevents.label.isin(Cefazolin)        |
                          inputevents.label.isin(Cefepime)             | inputevents.label.isin(Ceftazidime)      |
                          inputevents.label.isin(Vancomycin)           | inputevents.label.isin(Clindamycin)      |
                          inputevents.label.isin(Metronidazole)        | inputevents.label.isin(Meropenem)        |
                          inputevents.label.isin(Acyclovir)            | inputevents.label.isin(Azithromycin)     |
                          inputevents.label.isin(Levofloxacin)         | inputevents.label.isin(Micafungin)       |
                          inputevents.label.isin(Fluconazole)          | inputevents.label.isin(Thiamine)         |
                          inputevents.label.isin(Dobutamine)           | inputevents.label.isin(Milrinone)        |
                          inputevents.label.isin(Fluids)               | inputevents.label.isin(OralIntake)       |
                          inputevents.label.isin(PO)                   | inputevents.label.isin(SodiumChloride)   |
                          inputevents.label.isin(IVPB)                 | inputevents.label.isin(Stool)            |
                          inputevents.label.isin(Crystalloids)         | inputevents.label.isin(NSIVF)            |
                          inputevents.label.isin(Norepinephrine)       | inputevents.label.isin(Amiodarone)       |
                          inputevents.label.isin(Phenylephrine)        | inputevents.label.isin(Epinephrine)      |
                          inputevents.label.isin(Nicardipine)          | inputevents.label.isin(Pantoprazole)     |
                          inputevents.label.isin(Diltiazem)            | inputevents.label.isin(Nitroglycerin) ] 

In [121]:
inputevents.loc[inputevents.itemid.isin(Dobutamine_ID),      'label'] = 'Dobutamine_IO'
inputevents.loc[inputevents.itemid.isin(Milrinone_ID),       'label'] = 'Milrinone_IO'
inputevents.loc[inputevents.itemid.isin(NeuroblockAgent_ID), 'label'] = 'NeuroblockAgent_IO'
inputevents.loc[inputevents.itemid.isin(KIV_ID),             'label'] = 'K-IV_IO'
inputevents.loc[inputevents.itemid.isin(CaIV_ID),            'label'] = 'Ca-IV_IO'
inputevents.loc[inputevents.itemid.isin(CanonIV_ID),         'label'] = 'Ca-nonIV_IO'
inputevents.loc[inputevents.itemid.isin(MgIV_ID),            'label'] = 'Mg-IV_IO'
inputevents.loc[inputevents.itemid.isin(MgnonIV_ID),         'label'] = 'Mg-nonIV_IO'
inputevents.loc[inputevents.itemid.isin(PIV_ID),             'label'] = 'P-IV_IO'
inputevents.loc[inputevents.itemid.isin(PnonIV_ID),          'label'] = 'P-nonIV_IO'
inputevents.loc[inputevents.itemid.isin(Fluids_ID),          'label'] = 'Fluids_IO'
inputevents.loc[inputevents.itemid.isin(BetaBlockers_ID),    'label'] = 'BetaBlockers_IO'
inputevents.loc[inputevents.itemid.isin(CaBlockers_ID),      'label'] = 'CaBlockers_IO'
inputevents.loc[inputevents.itemid.isin(LoopDiuretics_ID),   'label'] = 'LoopDiuretics_IO'
inputevents.loc[inputevents.itemid.isin(TPNutrition_ID),     'label'] = 'TPNutrition_IO'
inputevents.loc[inputevents.itemid.isin(Fentanyl_ID),        'label'] = 'Fentanyl_IO'
inputevents.loc[inputevents.itemid.isin(Albumin_ID),         'label'] = 'Albumin_IO'
inputevents.loc[inputevents.itemid.isin(Insulin_ID),         'label'] = 'Insulin_IO'
inputevents.loc[inputevents.itemid.isin(Dextrose_ID),        'label'] = 'Dextrose_IO'
inputevents.loc[inputevents.itemid.isin(PNutrition_ID),      'label'] = 'PNutrition_IO'
inputevents.loc[inputevents.itemid.isin(POnutrition_ID),     'label'] = 'POnutrition_IO'
inputevents.loc[inputevents.itemid.isin(Vasopressors_ID),    'label'] = 'Vasopressors_IO'

In [122]:
inputevents.loc[inputevents.label.isin(Propofol)  ,       'label'] = 'Propofol_IO'
inputevents.loc[inputevents.label.isin(Fentanyl)  ,       'label'] = 'Fentanyl_IO'
inputevents.loc[inputevents.label.isin(Insulin)  ,        'label'] = 'Insulin_IO'
inputevents.loc[inputevents.label.isin(Heparin)  ,        'label'] = 'Heparin_IO'
inputevents.loc[inputevents.label.isin(Midazolam)  ,      'label'] = 'Midazolam_IO'
inputevents.loc[inputevents.label.isin(Dexmedetomidine) , 'label'] = 'Dexmedetomidine_IO'
inputevents.loc[inputevents.label.isin(Vassopressin)  ,   'label'] = 'Vassopressin_IO'
inputevents.loc[inputevents.label.isin(Albumin)  ,        'label'] = 'Albumin_IO'
inputevents.loc[inputevents.label.isin(Ceftriaxone)  ,    'label'] = 'Ceftriaxone_IO'
inputevents.loc[inputevents.label.isin(Cefazolin)  ,      'label'] = 'Cefazolin_IO'
inputevents.loc[inputevents.label.isin(Cefepime)  ,       'label'] = 'Cefepime_IO'
inputevents.loc[inputevents.label.isin(Ceftazidime)  ,    'label'] = 'Ceftazidime_IO'
inputevents.loc[inputevents.label.isin(Vancomycin)  ,     'label'] = 'Vancomycin_IO'
inputevents.loc[inputevents.label.isin(Clindamycin)  ,    'label'] = 'Clindamycin_IO'
inputevents.loc[inputevents.label.isin(Metronidazole)  ,  'label'] = 'Metronidazole_IO'
inputevents.loc[inputevents.label.isin(Meropenem)  ,      'label'] = 'Meropenem_IO'
inputevents.loc[inputevents.label.isin(Acyclovir)  ,      'label'] = 'Acyclovir_IO'
inputevents.loc[inputevents.label.isin(Azithromycin)  ,   'label'] = 'Azithromycin_IO'
inputevents.loc[inputevents.label.isin(Levofloxacin)  ,   'label'] = 'Levofloxacin_IO'
inputevents.loc[inputevents.label.isin(Micafungin)  ,     'label'] = 'Micafungin_IO'
inputevents.loc[inputevents.label.isin(Fluconazole)  ,    'label'] = 'Fluconazole_IO'
inputevents.loc[inputevents.label.isin(Thiamine)  ,       'label'] = 'Thiamine_IO'
inputevents.loc[inputevents.label.isin(Dobutamine)  ,     'label'] = 'Dobutamine_IO'
inputevents.loc[inputevents.label.isin(Milrinone)  ,      'label'] = 'Milrinone_IO'
inputevents.loc[inputevents.label.isin(Fluids)  ,         'label'] = 'Fluids_IO'
inputevents.loc[inputevents.label.isin(OralIntake)  ,     'label'] = 'OralIntake_IO'
inputevents.loc[inputevents.label.isin(PO)  ,             'label'] = 'P.O._IO'
inputevents.loc[inputevents.label.isin(SodiumChloride)  , 'label'] = 'SodiumChloride_IO'
inputevents.loc[inputevents.label.isin(IVPB)  ,           'label'] = 'IVPB_IO'
inputevents.loc[inputevents.label.isin(Stool)  ,          'label'] = 'Stool_IO'
inputevents.loc[inputevents.label.isin(Crystalloids)  ,   'label'] = 'Crystalloids_IO'
inputevents.loc[inputevents.label.isin(NSIVF)  ,          'label'] = 'NSIVF_IO'
inputevents.loc[inputevents.label.isin(Norepinephrine)  , 'label'] = 'Norepinephrine_IO'
inputevents.loc[inputevents.label.isin(Amiodarone)  ,     'label'] = 'Amiodarone_IO'
inputevents.loc[inputevents.label.isin(Phenylephrine)  ,  'label'] = 'Phenylephrine_IO'
inputevents.loc[inputevents.label.isin(Epinephrine)  ,    'label'] = 'Epinephrine_IO'
inputevents.loc[inputevents.label.isin(Nicardipine)  ,    'label'] = 'Nicardipine_IO'
inputevents.loc[inputevents.label.isin(Pantoprazole)  ,   'label'] = 'Pantoprazole_IO'
inputevents.loc[inputevents.label.isin(Diltiazem)  ,      'label'] = 'Diltiazem_IO'
inputevents.loc[inputevents.label.isin(Nitroglycerin)  ,  'label'] = 'Nitroglycerin_IO'

In [123]:
inputevents.drop(columns=['itemid', 'amountuom'], inplace=True)
inputevents = inputevents[['subject_id', 'hadm_id', 'stay_id', 'starttime', 'label', 'amount']]
inputevents.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']

In [124]:
inputevents.head(2)

In [125]:
print(inputevents.subject_id.nunique())
print(inputevents.hadm_id.nunique())
print(inputevents.stay_id.nunique())
print(inputevents.label.nunique())

16474
21184
23350
44


## Note

### radiology note

In [126]:
radioevent = []

for chunk in pd.read_csv(mimiciv + "note/radiology.csv", chunksize=10000):
    chunk.drop(columns=['storetime', 'text'], inplace=True)
    chunk = chunk[chunk.subject_id.isin(cohort_subject_id_icu)]

    if chunk.shape[0] > 0:
        radioevent.append(chunk)
        
radiology_note = pd.concat(radioevent)

In [127]:
radiology_note = pd.merge(radiology_note, cohort_icu_df, on=['subject_id'], how='left')
radiology_note['charttime'] = pd.to_datetime(radiology_note['charttime'])
radiology_note = radiology_note[(radiology_note['charttime'] >= radiology_note['intime']) & (radiology_note['charttime'] <= radiology_note['outtime'])]

In [128]:
radiology_note = radiology_note[['subject_id', 'hadm_id_y', 'stay_id', 'charttime', 'note_id']]
radiology_note.rename(columns={"hadm_id_y": "hadm_id"}, inplace=True)

In [129]:
radiology_note['label'] = 'radiology_note'
radiology_note = radiology_note[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'note_id']]
radiology_note.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']

In [130]:
radiology_note.head(3)

In [131]:
print(radiology_note.subject_id.nunique())
print(radiology_note.hadm_id.nunique())
print(radiology_note.stay_id.nunique())

13816
16855
18459


### discharge note

In [132]:
dischargevent = []

for chunk in pd.read_csv(mimiciv + "note/discharge.csv", chunksize=10000):
    chunk.drop(columns=['storetime', 'text'], inplace=True)
    chunk = chunk[chunk.hadm_id.isin(cohort_hadm_id_icu)]

    if chunk.shape[0] > 0:
        dischargevent.append(chunk)
        
discharge_note = pd.concat(dischargevent)

In [133]:
discharge_note = pd.merge(discharge_note, cohort_icu_df, on=['subject_id'], how='left')
discharge_note['charttime'] = pd.to_datetime(discharge_note['charttime'])
discharge_note = discharge_note[(discharge_note['charttime'] >= discharge_note['intime'])]

In [134]:
discharge_note = discharge_note[['subject_id', 'hadm_id_y', 'stay_id', 'charttime', 'note_id']]
discharge_note.rename(columns={"hadm_id_y": "hadm_id"}, inplace=True)

In [135]:
discharge_note['label'] = 'discharge_note'
discharge_note = discharge_note[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'note_id']]
discharge_note.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']

In [136]:
discharge_note.head(3)

In [137]:
print(discharge_note.subject_id.nunique())
print(discharge_note.hadm_id.nunique())
print(discharge_note.stay_id.nunique())

16234
20934
23196


## CXR

In [138]:
cxrmetadata = pd.read_csv(mimiciv + "cxr/mimic-cxr-2.0.0-metadata.csv")
cxrrecord   = pd.read_csv(mimiciv + "cxr/cxr-record-list.csv")
cxrstudy    = pd.read_csv(mimiciv + "cxr/cxr-study-list.csv")
lung_mask   = pd.read_csv(mimiciv + "cxr/lung_segmentation(mimic-cxr-1.0)/CXLSeg-mask.csv")

In [139]:
cxrrecord.rename(columns={"path": "path_image"}, inplace=True)
cxrstudy.rename(columns={"path": "path_note"}, inplace=True)
lung_mask.rename(columns={"DicomPath": "path_lung_seg"}, inplace=True)

In [140]:
cxr = pd.merge(cxrmetadata, cxrrecord, on=['dicom_id', 'subject_id', 'study_id'], how='left')
cxr = pd.merge(cxr, lung_mask, on=['subject_id', 'study_id', 'dicom_id'], how='left')
cxr = pd.merge(cxr, cxrstudy,  on=['subject_id', 'study_id'], how='left')
cxr = cxr[cxr.subject_id.isin(cohort_subject_id_icu)]

In [141]:
cxr['StudyDate'] = pd.to_datetime(cxr['StudyDate'], format='%Y%m%d')
cxr['StudyTime'] = cxr.apply(lambda x : '%#010.3f' % x['StudyTime'] ,1)
cxr['StudyTime'] = pd.to_datetime(cxr['StudyTime'], format='%H%M%S.%f').dt.time
cxr['cxrtime']   = cxr.apply(lambda r : dt.datetime.combine(r['StudyDate'],r['StudyTime']),1)
cxr['cxrtime']   = cxr['cxrtime'].dt.floor('Min')

In [142]:
cxr = pd.merge(cxr, cohort_icu_df, on=['subject_id'], how='left')
cxr = cxr[(cxr['cxrtime'] >= cxr['intime']) & (cxr['cxrtime'] <= cxr['outtime'])]
cxr = cxr[['subject_id', 'hadm_id', 'stay_id', 'study_id', 'dicom_id', 'cxrtime', 
           'ProcedureCodeSequence_CodeMeaning', 'ViewPosition', 'path_image', 'path_note', 'path_lung_seg']]

In [143]:
cxr.head(2)

In [144]:
print(cxr.subject_id.nunique())
print(cxr.study_id.nunique())
print(cxr.dicom_id.nunique())

3693
16783
19113


In [145]:
cxr_image = cxr[['subject_id', 'hadm_id', 'stay_id', 'cxrtime', 'dicom_id', 'path_image']].copy()
cxr_note  = cxr[['subject_id', 'hadm_id', 'stay_id', 'cxrtime', 'study_id', 'path_note']].copy()
cxr_lung  = cxr[['subject_id', 'hadm_id', 'stay_id', 'cxrtime', 'dicom_id', 'path_lung_seg']].copy()

cxr_image['label'] = 'cxr_image'
cxr_image = cxr_image[cxr_image.path_image.notnull()]
cxr_image = cxr_image[['subject_id', 'hadm_id', 'stay_id', 'cxrtime', 'label', 'dicom_id']]
cxr_image.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']

cxr_note['label'] = 'cxr_note'
cxr_note = cxr_note[cxr_note.path_note.notnull()]
cxr_note = cxr_note[['subject_id', 'hadm_id', 'stay_id', 'cxrtime', 'label', 'study_id']]
cxr_note.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']

cxr_lung['label'] = 'cxr_lung'
cxr_lung = cxr_lung[cxr_lung.path_lung_seg.notnull()]
cxr_lung = cxr_lung[['subject_id', 'hadm_id', 'stay_id', 'cxrtime', 'label', 'dicom_id']]
cxr_lung.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']

cxr_note = cxr_note.drop_duplicates()
cxr_lung = cxr_lung.drop_duplicates()

In [146]:
cxr_image.head(2)

In [147]:
print(cxr_image.subject_id.nunique())
print(cxr_image.hadm_id.nunique())
print(cxr_image.stay_id.nunique())
print(cxr_image.value.nunique())

3693
4370
4737
19113


In [148]:
cxr_note.head(2)

In [149]:
print(cxr_note.subject_id.nunique())
print(cxr_note.hadm_id.nunique())
print(cxr_note.stay_id.nunique())
print(cxr_note.value.nunique())

3693
4370
4737
16783


In [150]:
cxr_lung.head(2)

In [151]:
print(cxr_lung.subject_id.nunique())
print(cxr_lung.hadm_id.nunique())
print(cxr_lung.stay_id.nunique())
print(cxr_lung.value.nunique())

3606
4248
4607
18683


### ECG

In [152]:
ecg = pd.read_csv(mimiciv + "ecg/record_list.csv")
ecg_report = pd.read_csv(mimiciv + "ecg/waveform_note_links.csv")

In [153]:
ecg = ecg[['subject_id', 'study_id', 'ecg_time', 'file_name', 'path']]
ecg_report = ecg_report[['subject_id', 'study_id', 'charttime', 'note_id', 'waveform_path']]

ecg = ecg[ecg.subject_id.isin(cohort_subject_id_icu)]
ecg_report = ecg_report[ecg_report.subject_id.isin(cohort_subject_id_icu)]

ecg = pd.merge(ecg, cohort_icu_df, on=['subject_id'], how='left')
ecg_report = pd.merge(ecg_report, cohort_icu_df, on=['subject_id'], how='left')

ecg = ecg[(ecg['ecg_time'] >= ecg['intime']) & (ecg['ecg_time'] <= ecg['outtime'])]
ecg_report = ecg_report[(ecg_report['charttime'] >= ecg_report['intime']) & (ecg_report['charttime'] <= ecg_report['outtime'])]

ecg = ecg[['subject_id', 'hadm_id', 'stay_id', 'ecg_time', 'path']]
ecg_report = ecg_report[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'note_id']]

ecg['label'] = 'ecg_waveform'
ecg = ecg[ecg.path.notnull()]
ecg = ecg[['subject_id', 'hadm_id', 'stay_id', 'ecg_time', 'label', 'path']]
ecg.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']
ecg = ecg.drop_duplicates()

ecg_report['label'] = 'ecg_report'
ecg_report = ecg_report[ecg_report.note_id.notnull()]
ecg_report = ecg_report[['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'note_id']]
ecg_report.columns = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'label', 'value']
ecg_report = ecg_report.drop_duplicates()

In [154]:
ecg.head(3)

In [155]:
print(ecg.subject_id.nunique())
print(ecg.hadm_id.nunique())
print(ecg.stay_id.nunique())
print(ecg.value.nunique())

9281
10909
11580
22830


In [156]:
ecg_report.head(3)

In [157]:
print(ecg_report.subject_id.nunique())
print(ecg_report.hadm_id.nunique())
print(ecg_report.stay_id.nunique())
print(ecg_report.value.nunique())

7929
9494
10056
19420


## Save csv files

In [158]:
cohort_df.to_csv(output + 'demographic.csv'  , index=False)
prescriptions.to_csv(output + 'prescriptions.csv', index=False)
lab_df.to_csv(output    + 'lab_event.csv'  , index=False)
chart_df.to_csv(output  + 'chart_event.csv', index=False)
culture_df.to_csv(output   + 'culture_event.csv' , index=False)
microbio_df.to_csv(output  + 'microbio_event.csv', index=False)
datetimeevents.to_csv(output  + 'datetimeevents.csv', index=False)
outputevents.to_csv(output    + 'outputevents.csv'  , index=False)
inputevents.to_csv(output     + 'inputevents.csv'   , index=False)
radiology_note.to_csv(output  + 'radiology_note.csv', index=False)
discharge_note.to_csv(output  + 'discharge_note.csv', index=False)
cxr_image.to_csv(output  + 'cxr_image.csv', index=False)
cxr_note.to_csv(output   + 'cxr_note.csv' , index=False)
cxr_lung.to_csv(output   + 'cxr_lung.csv' , index=False)
ecg.to_csv(output   + 'ecg.csv' , index=False)
ecg_report.to_csv(output   + 'ecg_report.csv' , index=False)

In [21]:
with open(output + 'race_dictionary.pkl', 'wb') as f:
    pickle.dump(race_dictionary, f)